# Pix2Pix / Attention U-Net — Medical Image Translation
## Adapted for MMIT-DDPM BraTS Data

Trains a conditional image-to-image translation model on pre-processed MMIT-DDPM BraTS 2D slices.
Three architectures:
- `unet` — Vanilla U-Net (regression, no adversarial loss)
- `pix2pix` — U-Net generator + 70×70 PatchGAN (Isola et al. 2017)
- `attn_cgan` — Attention U-Net (Oktay et al. 2018) + multi-scale PatchGAN *(default)*

**Data**: `.nii.gz` 2D slices already normalized to [-1, 1] by the MMIT-DDPM pipeline.  
Naming: `brats_{split}_{patient_id}_{pair}_{slice}_w.nii.gz`  
Shape `(H, W, 2)` — channel 0 = source, channel 1 = target.

Available pairs: `ft1ce` `ft2` `t1cet2` `t1cetf` `t2f` `t2t1ce`

**No download or preprocessing required** — data lives in `./data/training/` and `./data/testing/`.

In [ ]:
# Run once; restart kernel after on Colab
!pip install -q nibabel scikit-image tqdm pytorch_msssim torchvision

In [ ]:
from __future__ import annotations

import os, random, warnings
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Literal, Optional, Tuple

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd
from skimage.metrics import peak_signal_noise_ratio as skimage_psnr
from skimage.metrics import structural_similarity as skimage_ssim
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

warnings.filterwarnings('ignore')

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.backends.cudnn.benchmark        = True
torch.set_float32_matmul_precision('high')

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'figure.facecolor': 'white'})

print(f'PyTorch : {torch.__version__}')
print(f'Device  : {DEVICE}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU     : {p.name}  |  VRAM: {p.total_memory / 1e9:.1f} GB')

## Configuration

In [ ]:
@dataclass
class Config:
    # Paths
    project_root : Path = Path('.')
    train_dir    : Path = Path('./data/training')
    test_dir     : Path = Path('./data/testing')

    # Translation pair — which modality pair to train
    # Options: ft1ce | ft2 | t1cet2 | t1cetf | t2f | t2t1ce
    translation_pair : str = 'ft1ce'   # Flair -> T1CE
    in_c             : int = 1         # source channels (1 per pair)

    # Splits
    train_frac       : float = 0.80
    split_seed       : int   = 42
    dataset_fraction : float = 1.0    # < 1.0 for smoke-tests

    # Model: 'unet' | 'pix2pix' | 'attn_cgan'
    model_type : Literal['unet', 'pix2pix', 'attn_cgan'] = 'attn_cgan'
    feats      : Tuple[int, ...] = (64, 128, 256, 512)
    dropout    : float = 0.3
    ndf        : int   = 64

    # Training
    max_epochs        : int   = 200
    batch_size        : int   = 16
    lr_g              : float = 2e-4
    lr_d              : float = 1e-4
    n_gen_steps       : int   = 2
    grad_clip         : float = 1.0
    val_every         : int   = 5
    patience          : int   = 15
    num_workers       : int   = 2
    tumour_oversample : float = 3.0

    # Loss weights
    w_adv         : float = 1.0
    w_l1          : float = 100.0
    w_perc        : float = 10.0
    w_freq        : float = 5.0
    w_ssim        : float = 10.0
    lambda_tumour : float = 5.0

    # EMA
    ema_decay : float = 0.999

    # Augmentation
    aug_flip_p      : float = 0.5
    aug_vflip_p     : float = 0.5
    aug_rot_p       : float = 0.5
    aug_max_angle   : float = 15.0
    aug_intensity_p : float = 0.5

    # Metrics — data normalized to [-1, 1] => fixed range = 2.0
    metric_data_range : float = 2.0

    # Output
    out_name : str = 'run_ft1ce_attn_cgan'

    @property
    def splits_csv(self) -> Path: return self.project_root / 'splits.csv'
    @property
    def out_dir(self)    -> Path: return self.project_root / 'outputs' / self.out_name


CFG = Config()
print('-' * 55)
print(f'  pair             : {CFG.translation_pair}')
print(f'  model_type       : {CFG.model_type}')
print(f'  in_c             : {CFG.in_c}')
print(f'  dataset_fraction : {CFG.dataset_fraction:.0%}')
print(f'  batch_size       : {CFG.batch_size}')
print(f'  max_epochs       : {CFG.max_epochs}')
print(f'  train_dir        : {CFG.train_dir}')
print(f'  test_dir         : {CFG.test_dir}')
print(f'  out_dir          : {CFG.out_dir}')
print('-' * 55)

## Data Pipeline

`MMITBraTS2DDataset` reads `.nii.gz` files directly — **no preprocessing step needed**.  
Since paired slice files do not include segmentation masks, a binary brain mask
(non-zero target voxels) is used as a surrogate for the tumour mask in the
weighted L1 loss term.

In [ ]:
# Human-readable names for each translation pair
PAIR_LABELS: Dict[str, Tuple[str, str]] = {
    'ft1ce' : ('Flair', 'T1CE'),
    'ft2'   : ('Flair', 'T2'),
    't1cet2': ('T1CE', 'T2'),
    't1cetf': ('T1CE', 'Flair'),
    't2f'   : ('T2',   'Flair'),
    't2t1ce': ('T2',   'T1CE'),
}


def parse_mmit_filename(fname: str) -> Optional[Dict[str, str]]:
    """Parse brats_{split}_{patient_id}_{pair}_{slice}_w.nii.gz into components."""
    stem  = fname.replace('.nii.gz', '').replace('_w', '')
    parts = stem.split('_')
    if len(parts) < 5 or parts[0] != 'brats':
        return None
    for i, p in enumerate(parts):
        if p in PAIR_LABELS:
            return {
                'split':      parts[1],
                'patient_id': '_'.join(parts[2:i]),
                'pair':       p,
                'slice':      parts[i + 1],
            }
    return None


class PairedAugmenter:
    """
    Paired geometric + intensity augmentation.
    Geometric transforms are applied jointly to source, target, and mask.
    Intensity jitter is applied to the source only.
    Works for any in_c (source channel count).
    """

    def __init__(self, cfg: Config) -> None:
        self.flip_p   = cfg.aug_flip_p
        self.vflip_p  = cfg.aug_vflip_p
        self.rot_p    = cfg.aug_rot_p
        self.max_ang  = cfg.aug_max_angle
        self.intens_p = cfg.aug_intensity_p

    def _intensity_jitter(self, x: torch.Tensor) -> torch.Tensor:
        x    = x + random.uniform(-0.15, 0.15)
        mu   = x.mean()
        x    = (x - mu) * random.uniform(0.8, 1.2) + mu
        g    = random.uniform(0.8, 1.2)
        x01  = ((x.clamp(-1, 1) + 1.0) / 2.0).clamp(0, 1)
        return (x01.pow(g) * 2.0 - 1.0).clamp(-1, 1)

    def __call__(self, x, y, seg):
        import torchvision.transforms.functional as TF
        in_c     = x.shape[0]
        combined = torch.cat([x, y, seg.float()], dim=0)  # (in_c+2, H, W)
        if random.random() < self.flip_p:
            combined = torch.flip(combined, dims=[2])
        if random.random() < self.vflip_p:
            combined = torch.flip(combined, dims=[1])
        if random.random() < self.rot_p:
            combined = TF.rotate(combined, random.uniform(-self.max_ang, self.max_ang),
                                 interpolation=TF.InterpolationMode.BILINEAR)
        x_out   = combined[:in_c]
        y_out   = combined[in_c:in_c + 1]
        seg_out = (combined[in_c + 1:in_c + 2] > 0.5).float()
        if random.random() < self.intens_p:
            x_out = self._intensity_jitter(x_out)
        return x_out, y_out, seg_out


class MMITBraTS2DDataset(Dataset):
    """
    Reads MMIT-DDPM NIfTI paired slices directly from disk.

    Each .nii.gz: shape (H, W, 2), pre-normalized to [-1, 1].
      channel 0 = source modality
      channel 1 = target modality

    Returns
    -------
    x    : (in_c, H, W) float32  source
    y    : (1, H, W)    float32  target
    seg  : (1, H, W)    float32  brain mask (surrogate for tumour mask)
    meta : dict  {pid, z, has_tumour}
    """

    def __init__(self, data_dir, patient_ids, pair, augment=False, cfg=None):
        self.augmenter = PairedAugmenter(cfg) if (augment and cfg) else None
        pid_set        = set(patient_ids)
        self.files: List[Path] = []
        self.meta_list: List[dict] = []

        for f in sorted(Path(data_dir).glob(f'*_{pair}_*_w.nii.gz')):
            info = parse_mmit_filename(f.name)
            if info and info['patient_id'] in pid_set:
                self.files.append(f)
                self.meta_list.append({'pid': info['patient_id'], 'z': int(info['slice'])})

        # All BraTS slices originate from tumour patients
        self.has_tumour_flags = [True] * len(self.files)
        self.tumour_fracs     = [1.0]  * len(self.files)

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        vol = nib.load(str(self.files[idx])).get_fdata().astype(np.float32)
        src = torch.from_numpy(vol[:, :, 0]).unsqueeze(0)  # (1, H, W)
        tgt = torch.from_numpy(vol[:, :, 1]).unsqueeze(0)  # (1, H, W)
        seg = (tgt.abs() > 0.05).float()                   # brain mask
        if self.augmenter:
            src, tgt, seg = self.augmenter(src, tgt, seg)
        meta = {'pid': self.meta_list[idx]['pid'],
                'z':   self.meta_list[idx]['z'],
                'has_tumour': True}
        return src, tgt, seg, meta


print('Data classes defined.')

In [ ]:
class SplitManager:
    """
    Patient-level 80/20 (train/val) split from ./data/training/
    plus a separate test set from ./data/testing/.
    Supports dataset_fraction < 1.0 for rapid smoke-tests.
    """

    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg

    def _discover_patients(self, data_dir: Path) -> List[str]:
        pids: set = set()
        for f in Path(data_dir).glob(f'*_{self.cfg.translation_pair}_*_w.nii.gz'):
            info = parse_mmit_filename(f.name)
            if info:
                pids.add(info['patient_id'])
        return sorted(pids)

    def make(self, force: bool = False) -> None:
        if self.cfg.splits_csv.exists() and not force:
            print(f'[SplitManager] {self.cfg.splits_csv} already exists (use force=True to regenerate).')
            return
        train_pids = self._discover_patients(self.cfg.train_dir)
        test_pids  = self._discover_patients(self.cfg.test_dir)
        if not train_pids:
            raise RuntimeError(
                f"No files in {self.cfg.train_dir} for pair '{self.cfg.translation_pair}'."
            )
        rng  = np.random.default_rng(self.cfg.split_seed)
        pids = list(rng.permutation(train_pids))
        t    = max(1, int(len(pids) * self.cfg.train_frac))
        rows = (
            [{'patient_id': p, 'split': 'train', 'data_dir': str(self.cfg.train_dir)} for p in pids[:t]]
          + [{'patient_id': p, 'split': 'val',   'data_dir': str(self.cfg.train_dir)} for p in pids[t:]]
          + [{'patient_id': p, 'split': 'test',  'data_dir': str(self.cfg.test_dir)}  for p in test_pids]
        )
        pd.DataFrame(rows).to_csv(self.cfg.splits_csv, index=False)
        print(f'  train: {t} | val: {len(pids)-t} | test: {len(test_pids)}')

    def load(self) -> Dict[str, Dict]:
        if not self.cfg.splits_csv.exists():
            self.make()
        df     = pd.read_csv(self.cfg.splits_csv)
        splits = defaultdict(lambda: {'pids': [], 'data_dir': None})
        for _, row in df.iterrows():
            splits[row['split']]['pids'].append(row['patient_id'])
            splits[row['split']]['data_dir'] = Path(row['data_dir'])

        frac = self.cfg.dataset_fraction
        if frac < 1.0:
            rng = np.random.default_rng(self.cfg.split_seed + 1)
            for key in splits:
                pids = list(rng.permutation(splits[key]['pids']))
                splits[key]['pids'] = pids[:max(1, int(len(pids) * frac))]

        for k, v in splits.items():
            print(f'  [{k:5s}]  {len(v["pids"])} patients  |  {v["data_dir"]}')
        return dict(splits)


class DataLoaderFactory:
    def __init__(self, cfg: Config) -> None:
        self.cfg = cfg

    def build(self, splits: Dict[str, Dict]) -> Dict[str, DataLoader]:
        loaders = {}
        for split_name, info in splits.items():
            is_train = split_name == 'train'
            ds = MMITBraTS2DDataset(
                data_dir    = info['data_dir'],
                patient_ids = info['pids'],
                pair        = self.cfg.translation_pair,
                augment     = is_train,
                cfg         = self.cfg,
            )
            if len(ds) == 0:
                print(f'  [{split_name:5s}]  WARNING: 0 slices found.')
                continue
            kw = dict(batch_size=self.cfg.batch_size, num_workers=self.cfg.num_workers,
                      pin_memory=True,
                      persistent_workers=self.cfg.num_workers > 0,
                      prefetch_factor=2 if self.cfg.num_workers > 0 else None)
            if is_train:
                weights = [1.0 + (self.cfg.tumour_oversample - 1.0) * min(tf * 20.0, 1.0)
                           for tf in ds.tumour_fracs]
                loaders[split_name] = DataLoader(
                    ds, sampler=WeightedRandomSampler(weights, len(weights), replacement=True),
                    drop_last=True, **kw)
            else:
                loaders[split_name] = DataLoader(ds, shuffle=False, **kw)
            print(f'  [{split_name:5s}]  {len(info["pids"])} patients  |  {len(ds):,} slices')
        return loaders


print('SplitManager and DataLoaderFactory defined.')

## Model Architectures

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_c, out_c, norm='instance', dropout=0.0):
        super().__init__()
        def _n(c): return nn.InstanceNorm2d(c, affine=True) if norm == 'instance' else nn.GroupNorm(min(8, c), c)
        layers = [nn.Conv2d(in_c,  out_c, 3, padding=1, bias=False), _n(out_c), nn.LeakyReLU(0.2, True),
                  nn.Conv2d(out_c, out_c, 3, padding=1, bias=False), _n(out_c), nn.LeakyReLU(0.2, True)]
        if dropout > 0.0: layers.append(nn.Dropout2d(dropout))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


class AttentionGate(nn.Module):
    """Soft spatial attention gate (Oktay et al. 2018)."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g  = nn.Sequential(nn.Conv2d(F_g, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.W_x  = nn.Sequential(nn.Conv2d(F_l, F_int, 1, bias=True), nn.BatchNorm2d(F_int))
        self.psi  = nn.Sequential(nn.Conv2d(F_int, 1, 1, bias=True),   nn.BatchNorm2d(1), nn.Sigmoid())
        self.relu = nn.ReLU(inplace=True)
    def forward(self, g, x):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        return x * self.psi(self.relu(g1 + x1))


class DiscBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=2, norm=True):
        super().__init__()
        layers = [nn.Conv2d(in_c, out_c, 4, stride=stride, padding=1, bias=not norm)]
        if norm: layers.append(nn.InstanceNorm2d(out_c, affine=True))
        layers.append(nn.LeakyReLU(0.2, True))
        self.block = nn.Sequential(*layers)
    def forward(self, x): return self.block(x)


print('DoubleConv, AttentionGate, DiscBlock defined.')

In [ ]:
class BaseUNet(nn.Module):
    def __init__(self, in_c=1, out_c=1, feats=(64,128,256,512), tanh=True, norm='instance'):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc  = nn.ModuleList()
        ch = in_c
        for f in feats:
            self.enc.append(DoubleConv(ch, f, norm=norm)); ch = f
        self.bottleneck = DoubleConv(ch, ch*2, norm=norm); ch = ch*2
        self.up_proj = nn.ModuleList(); self.dec = nn.ModuleList()
        for f in reversed(feats):
            self.up_proj.append(nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(ch, f, 1, bias=False)))
            self.dec.append(DoubleConv(f*2, f, norm=norm)); ch = f
        out = [nn.Conv2d(ch, out_c, 1)]
        if tanh: out.append(nn.Tanh())
        self.out_head = nn.Sequential(*out)

    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for up, d, s in zip(self.up_proj, self.dec, reversed(skips)):
            x = up(x)
            if x.shape[2:] != s.shape[2:]:
                x = F.interpolate(x, size=s.shape[2:], mode='bilinear', align_corners=False)
            x = d(torch.cat([s, x], dim=1))
        return self.out_head(x)

    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


class AttentionUNet(nn.Module):
    def __init__(self, in_c=1, out_c=1, feats=(64,128,256,512), dropout=0.3, norm='instance'):
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.enc  = nn.ModuleList()
        ch = in_c
        for f in feats:
            self.enc.append(DoubleConv(ch, f, norm=norm)); ch = f
        self.bottleneck = DoubleConv(ch, ch*2, norm=norm); ch = ch*2
        self.att_gates = nn.ModuleList(); self.up_proj = nn.ModuleList(); self.dec = nn.ModuleList()
        g_ch = ch
        for i, f in enumerate(reversed(feats)):
            self.att_gates.append(AttentionGate(F_g=g_ch, F_l=f, F_int=f//2))
            self.up_proj.append(nn.Sequential(
                nn.Upsample(scale_factor=2, mode='bilinear', align_corners=False),
                nn.Conv2d(g_ch, f, 1, bias=False)))
            self.dec.append(DoubleConv(f*2, f, norm=norm, dropout=dropout if i < 2 else 0.0))
            g_ch = f
        self.out_head = nn.Sequential(nn.Conv2d(g_ch, out_c, 1), nn.Tanh())

    def forward(self, x):
        skips = []
        for e in self.enc:
            x = e(x); skips.append(x); x = self.pool(x)
        x = self.bottleneck(x)
        for up, ag, d, s in zip(self.up_proj, self.att_gates, self.dec, reversed(skips)):
            sa = ag(x, s); x = up(x)
            if x.shape[2:] != sa.shape[2:]:
                x = F.interpolate(x, size=sa.shape[2:], mode='bilinear', align_corners=False)
            x = d(torch.cat([sa, x], dim=1))
        return self.out_head(x)

    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


def build_generator(cfg: Config) -> nn.Module:
    if cfg.model_type == 'unet':
        return BaseUNet(in_c=cfg.in_c, out_c=1, feats=cfg.feats, tanh=True)
    return AttentionUNet(in_c=cfg.in_c, out_c=1, feats=cfg.feats, dropout=cfg.dropout)


for mt in ('unet', 'attn_cgan'):
    _c = Config(model_type=mt)
    _g = build_generator(_c)
    with torch.no_grad():
        _o = _g(torch.randn(2, _c.in_c, 240, 240))
    print(f'{mt:12s}  params={_g.n_params:,}  out={tuple(_o.shape)}')
    del _g, _o

In [ ]:
class PatchGANDiscriminator(nn.Module):
    """Conditional PatchGAN (Isola et al. 2017). 70x70 receptive field."""
    def __init__(self, in_c=1, ndf=64):
        super().__init__()
        self.model = nn.Sequential(
            DiscBlock(in_c+1, ndf,    stride=2, norm=False),
            DiscBlock(ndf,    ndf*2,  stride=2, norm=True),
            DiscBlock(ndf*2,  ndf*4,  stride=2, norm=True),
            DiscBlock(ndf*4,  ndf*8,  stride=1, norm=True),
            nn.Conv2d(ndf*8, 1, kernel_size=4, stride=1, padding=1),
        )
    def forward(self, x, y): return self.model(torch.cat([x, y], dim=1))
    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


class MultiScaleDiscriminator(nn.Module):
    """Two PatchGANs at original and 2x downsampled resolution."""
    def __init__(self, in_c=1, ndf=64):
        super().__init__()
        self.D1   = PatchGANDiscriminator(in_c=in_c, ndf=ndf)
        self.D2   = PatchGANDiscriminator(in_c=in_c, ndf=ndf)
        self.down = nn.AvgPool2d(2, stride=2)
    def forward(self, x, y): return [self.D1(x, y), self.D2(self.down(x), self.down(y))]
    @property
    def n_params(self): return sum(p.numel() for p in self.parameters() if p.requires_grad)


def build_discriminator(cfg: Config):
    if cfg.model_type == 'unet':     return None
    if cfg.model_type == 'pix2pix': return PatchGANDiscriminator(in_c=cfg.in_c, ndf=cfg.ndf)
    return MultiScaleDiscriminator(in_c=cfg.in_c, ndf=cfg.ndf)


for mt, disc in [('pix2pix', PatchGANDiscriminator(1,64)), ('attn_cgan', MultiScaleDiscriminator(1,64))]:
    _x, _y = torch.randn(2,1,240,240), torch.randn(2,1,240,240)
    with torch.no_grad():
        _s = disc(_x, _y)
    shapes = [tuple(s.shape) for s in (_s if isinstance(_s, list) else [_s])]
    print(f'{mt:12s}  params={disc.n_params:,}  out={shapes}')
    del disc, _x, _y, _s

## Loss Functions

In [ ]:
def tumour_weighted_l1(pred, target, seg, lambda_t=5.0):
    if seg is None or lambda_t <= 1.0: return F.l1_loss(pred, target)
    weights = torch.ones_like(target) + (lambda_t - 1.0) * (seg > 0).float()
    return (weights * torch.abs(pred - target)).mean()


def freq_domain_loss(pred, gt):
    return F.l1_loss(
        torch.abs(torch.fft.rfft2(pred.float(), norm='ortho')),
        torch.abs(torch.fft.rfft2(gt.float(),   norm='ortho')),
    )


class PerceptualLoss(nn.Module):
    """VGG-16 feature-matching loss. Gracefully degrades if torchvision is unavailable."""
    _MEAN = [0.485, 0.456, 0.406]
    _STD  = [0.229, 0.224, 0.225]

    def __init__(self, layer_ids=(3, 8)):
        super().__init__()
        try:
            import torchvision.models as tvm
            vgg = tvm.vgg16(weights=tvm.VGG16_Weights.DEFAULT).features.eval()
            self.slices = nn.ModuleList()
            prev = 0
            for lid in layer_ids:
                self.slices.append(nn.Sequential(*list(vgg.children())[prev:lid+1])); prev = lid+1
            for p in self.parameters(): p.requires_grad_(False)
            self.register_buffer('mean', torch.tensor(self._MEAN).view(1,3,1,1))
            self.register_buffer('std',  torch.tensor(self._STD ).view(1,3,1,1))
            self._ok = True
        except Exception as e:
            print(f'[PerceptualLoss] VGG unavailable ({e}). Term will be 0.')
            self._ok = False

    def _prep(self, x):
        x = (x + 1.0) / 2.0
        if x.shape[1] == 1: x = x.repeat(1,3,1,1)
        return (x - self.mean) / self.std

    def forward(self, pred, gt):
        if not self._ok: return torch.tensor(0.0, device=pred.device)
        p, g = self._prep(pred), self._prep(gt)
        loss = torch.tensor(0.0, device=pred.device)
        for s in self.slices:
            p = s(p); g = s(g).detach(); loss = loss + F.l1_loss(p, g)
        return loss


def discriminator_hinge_loss(real_scores, fake_scores):
    loss = torch.tensor(0.0, device=real_scores[0].device)
    for r, f in zip(real_scores, fake_scores):
        loss = loss + F.relu(1.0 - r).mean() + F.relu(1.0 + f).mean()
    return loss / len(real_scores)


def generator_adversarial_loss(fake_scores):
    return sum(-f.mean() for f in fake_scores) / len(fake_scores)


try:
    from pytorch_msssim import ssim as _pms_ssim
    _SSIM_BACKEND = 'pytorch_msssim'
except ImportError:
    _SSIM_BACKEND = 'fallback'


def _gaussian_win(sz, sigma, device, dtype):
    c = torch.arange(sz, dtype=dtype, device=device) - sz // 2
    g = torch.exp(-(c**2) / (2*sigma**2)); g /= g.sum()
    return (g.unsqueeze(0) * g.unsqueeze(1)).unsqueeze(0).unsqueeze(0)


def _ssim_fallback(pred, gt, dr=2.0, ws=11, sigma=1.5):
    C1, C2 = (0.01*dr)**2, (0.03*dr)**2
    win = _gaussian_win(ws, sigma, pred.device, pred.dtype).expand(pred.shape[1], 1, ws, ws)
    pad = ws // 2
    def _f(x): return F.conv2d(x, win, padding=pad, groups=x.shape[1])
    mp, mg = _f(pred), _f(gt)
    sp2  = _f(pred*pred) - mp*mp; sg2 = _f(gt*gt) - mg*mg; spg = _f(pred*gt) - mp*mg
    return (((2*mp*mg+C1)*(2*spg+C2)) / ((mp*mp+mg*mg+C1)*(sp2+sg2+C2))).mean()


def ssim_loss(pred, gt, data_range=2.0):
    p32, g32 = pred.float(), gt.float()
    if _SSIM_BACKEND == 'pytorch_msssim':
        return 1.0 - _pms_ssim(p32, g32, data_range=data_range, size_average=True)
    return 1.0 - _ssim_fallback(p32, g32, dr=data_range)


class CompositeLoss(nn.Module):
    """
    L = w_adv*L_cGAN + w_l1*L1 + w_perc*L_VGG + w_freq*L_FFT + w_ssim*(1-SSIM)
    w_adv is automatically 0 for model_type='unet'.
    """
    def __init__(self, cfg):
        super().__init__()
        self.w_adv  = cfg.w_adv if cfg.model_type != 'unet' else 0.0
        self.w_l1   = cfg.w_l1
        self.w_perc = cfg.w_perc
        self.w_freq = cfg.w_freq
        self.w_ssim = cfg.w_ssim
        self.lt     = cfg.lambda_tumour
        self.dr     = cfg.metric_data_range
        self.vgg    = PerceptualLoss(layer_ids=(3, 8))

    def forward(self, fake_scores, pred, gt, seg=None):
        L = {}
        L['adv']  = (generator_adversarial_loss(fake_scores) * self.w_adv
                     if self.w_adv > 0 and fake_scores else torch.tensor(0.0, device=pred.device))
        L['l1']   = tumour_weighted_l1(pred, gt, seg, self.lt) * self.w_l1
        L['perc'] = self.vgg(pred, gt) * self.w_perc
        L['freq'] = freq_domain_loss(pred, gt) * self.w_freq
        L['ssim'] = ssim_loss(pred, gt, data_range=self.dr) * self.w_ssim
        total = sum(L.values())
        return total, {k: float(v.item()) for k, v in L.items()}


# Sanity check
_cfg = Config(model_type='attn_cgan')
_c   = CompositeLoss(_cfg)
_p   = torch.tanh(torch.randn(2,1,64,64))
_g   = torch.tanh(torch.randn(2,1,64,64))
_fs  = [torch.randn(2,1,8,8), torch.randn(2,1,4,4)]
_t, _d = _c(_fs, _p, _g)
print('CompositeLoss OK:', {k: f'{v:.3f}' for k,v in _d.items()}, f'total={_t.item():.3f}')
print(f'SSIM backend: {_SSIM_BACKEND}')
del _cfg, _c, _p, _g, _fs, _t, _d

## Training Engine

In [ ]:
class MetricsTracker:
    """Accumulates SSIM/PSNR/MAE with a fixed data_range."""
    def __init__(self, data_range=2.0):
        self.dr = data_range; self.records: List[dict] = []

    def reset(self): self.records.clear()

    def update(self, pred_np, gt_np, has_tumour):
        mask = gt_np != 0
        if mask.sum() == 0: return
        self.records.append({
            'ssim':       float(skimage_ssim(pred_np, gt_np, data_range=self.dr)),
            'psnr':       float(skimage_psnr(gt_np, pred_np, data_range=self.dr)),
            'mae':        float(np.abs(pred_np[mask] - gt_np[mask]).mean()),
            'has_tumour': has_tumour,
        })

    def _agg(self, sub, key):
        vals = [r[key] for r in sub if np.isfinite(r[key])]
        return {'mean': float(np.mean(vals)), 'std': float(np.std(vals)), 'n': len(vals)} if vals \
               else {'mean': float('nan'), 'std': float('nan'), 'n': 0}

    def summary(self):
        t = [r for r in self.records if     r['has_tumour']]
        h = [r for r in self.records if not r['has_tumour']]
        return {'all': {k: self._agg(self.records, k) for k in ('ssim','psnr','mae')},
                'tumour':  {k: self._agg(t, k) for k in ('ssim','psnr','mae')},
                'healthy': {k: self._agg(h, k) for k in ('ssim','psnr','mae')}}

    def ssim_mean(self):
        vals = [r['ssim'] for r in self.records if np.isfinite(r['ssim'])]
        return float(np.mean(vals)) if vals else 0.0


class EarlyStopping:
    def __init__(self, patience=15, min_delta=5e-4):
        self.patience = patience; self.min_delta = min_delta
        self.best = -np.inf; self.counter = 0; self.triggered = False
    def __call__(self, score):
        if score > self.best + self.min_delta:
            self.best = score; self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience: self.triggered = True
        return self.triggered


print('MetricsTracker and EarlyStopping defined.')

In [ ]:
class Trainer:
    """
    End-to-end loop for unet / pix2pix / attn_cgan.
    Features: AMP, gradient clipping, EMA weights, linear LR decay,
              best-checkpoint saving, training log CSV.
    """

    def __init__(self, cfg):
        self.cfg = cfg
        cfg.out_dir.mkdir(parents=True, exist_ok=True)
        self._ckpt     = cfg.out_dir / 'best.pth'
        self._log_path = cfg.out_dir / 'train_log.csv'

    def _init_models(self):
        gen  = build_generator(self.cfg).to(DEVICE)
        disc = build_discriminator(self.cfg)
        if disc is not None: disc = disc.to(DEVICE)
        def _init(m):
            if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
                nn.init.normal_(m.weight.data, 0.0, 0.02)
            elif isinstance(m, (nn.BatchNorm2d, nn.InstanceNorm2d)):
                if hasattr(m,'weight') and m.weight is not None: nn.init.normal_(m.weight.data, 1.0, 0.02)
                if hasattr(m,'bias')   and m.bias   is not None: nn.init.constant_(m.bias.data, 0.0)
        gen.apply(_init)
        if disc: disc.apply(_init)
        print(f'Generator     : {gen.n_params:,} params')
        if disc: print(f'Discriminator : {disc.n_params:,} params')
        if hasattr(torch, 'compile'):
            try:
                gen  = torch.compile(gen,  mode='reduce-overhead')
                if disc: disc = torch.compile(disc, mode='default')
                print('[Trainer] torch.compile active.')
            except Exception as e:
                print(f'[Trainer] torch.compile skipped: {e}')
        return gen, disc

    def _build_opts(self, gen, disc):
        opt_g = torch.optim.Adam(gen.parameters(),  lr=self.cfg.lr_g, betas=(0.5, 0.999))
        opt_d = torch.optim.Adam(disc.parameters(), lr=self.cfg.lr_d, betas=(0.5, 0.999)) if disc else None
        decay = self.cfg.max_epochs // 2
        lam   = lambda ep: 1.0 if ep < decay else max(0.0, 1.0 - (ep-decay)/decay)
        return opt_g, opt_d, \
               torch.optim.lr_scheduler.LambdaLR(opt_g, lam), \
               (torch.optim.lr_scheduler.LambdaLR(opt_d, lam) if opt_d else None)

    def _save(self, epoch, gen, disc, opt_g, opt_d, metrics, sg=None, sd=None, ema=None):
        sd_ = lambda m: (m._orig_mod if hasattr(m,'_orig_mod') else m).state_dict()
        p = {'epoch': epoch, 'gen': sd_(gen), 'metrics': metrics}
        if disc:  p['disc']    = sd_(disc)
        if opt_g: p['opt_g']   = opt_g.state_dict()
        if opt_d: p['opt_d']   = opt_d.state_dict()
        if sg:    p['scaler_g']= sg.state_dict()
        if sd:    p['scaler_d']= sd.state_dict()
        if ema:   p['gen_ema'] = sd_(ema)
        torch.save(p, self._ckpt)

    def load_checkpoint(self, gen, disc, opt_g=None, opt_d=None, sg=None, sd=None, ema=None):
        if not self._ckpt.exists(): print('[Trainer] No checkpoint.'); return 0
        c   = torch.load(self._ckpt, map_location=DEVICE, weights_only=False)
        tgt = lambda m: (m._orig_mod if hasattr(m,'_orig_mod') else m)
        tgt(gen).load_state_dict(c['gen'])
        if disc and 'disc' in c:    tgt(disc).load_state_dict(c['disc'])
        if opt_g and 'opt_g' in c:  opt_g.load_state_dict(c['opt_g'])
        if opt_d and 'opt_d' in c:  opt_d.load_state_dict(c['opt_d'])
        if sg and 'scaler_g' in c:  sg.load_state_dict(c['scaler_g'])
        if sd and 'scaler_d' in c:  sd.load_state_dict(c['scaler_d'])
        if ema and 'gen_ema' in c:  tgt(ema).load_state_dict(c['gen_ema'])
        ep = int(c.get('epoch', 0))
        print(f'[Trainer] Loaded epoch={ep}  SSIM={c["metrics"].get("ssim_all",0):.4f}')
        return ep

    @torch.no_grad()
    def _validate(self, gen, val_dl):
        gen.eval()
        tr = MetricsTracker(data_range=self.cfg.metric_data_range)
        for x, y, seg, meta in val_dl:
            x = x.to(DEVICE)
            with autocast(device_type=AMP_DEVICE):
                pred = gen(x)
            pn = pred.cpu().float().numpy(); yn = y.float().numpy()
            for b in range(pn.shape[0]):
                ht = bool(meta['has_tumour'][b]) if isinstance(meta['has_tumour'], torch.Tensor) \
                     else meta['has_tumour'][b]
                tr.update(pn[b,0], yn[b,0], ht)
        return tr.summary()

    def fit(self, loaders, resume=False):
        train_dl, val_dl = loaders['train'], loaders['val']
        gen, disc        = self._init_models()
        opt_g, opt_d, sch_g, sch_d = self._build_opts(gen, disc)
        crit    = CompositeLoss(self.cfg).to(DEVICE)
        sg, sd  = GradScaler(device=AMP_DEVICE), GradScaler(device=AMP_DEVICE)
        stopper = EarlyStopping(patience=self.cfg.patience)
        best_ssim, best_m = -np.inf, {}
        log_rows: List[dict] = []
        start_ep = 1

        # EMA copy of generator
        ema = build_generator(self.cfg).to(DEVICE)
        _tgt = lambda m: (m._orig_mod if hasattr(m,'_orig_mod') else m)
        _tgt(ema).load_state_dict(_tgt(gen).state_dict())
        for p in ema.parameters(): p.requires_grad_(False)
        dc = self.cfg.ema_decay

        @torch.no_grad()
        def _ema_update():
            src, dst = _tgt(gen).state_dict(), _tgt(ema).state_dict()
            for k in dst:
                if dst[k].dtype.is_floating_point: dst[k].mul_(dc).add_(src[k], alpha=1-dc)
                else: dst[k].copy_(src[k])

        if resume:
            start_ep = self.load_checkpoint(gen, disc, opt_g=opt_g, opt_d=opt_d,
                                            sg=sg, sd=sd, ema=ema) + 1

        for epoch in range(start_ep, self.cfg.max_epochs + 1):
            gen.train()
            if disc: disc.train()
            sum_g = sum_d = 0.0; n = 0
            pbar = tqdm(train_dl, desc=f'Ep {epoch:3d}/{self.cfg.max_epochs}', ncols=100, leave=False)

            for x, y, seg, _ in pbar:
                x, y, seg = x.to(DEVICE), y.to(DEVICE), seg.to(DEVICE)

                if disc and opt_d:
                    with autocast(device_type=AMP_DEVICE):
                        yf = gen(x)
                        rs = disc(x, y);            rs = rs if isinstance(rs, list) else [rs]
                        fs = disc(x, yf.detach()); fs = fs if isinstance(fs, list) else [fs]
                        ld = discriminator_hinge_loss(rs, fs)
                    opt_d.zero_grad(set_to_none=True)
                    sd.scale(ld).backward(); sd.unscale_(opt_d)
                    torch.nn.utils.clip_grad_norm_(disc.parameters(), self.cfg.grad_clip)
                    sd.step(opt_d); sd.update(); sum_d += ld.item()

                for _ in range(self.cfg.n_gen_steps if disc else 1):
                    with autocast(device_type=AMP_DEVICE):
                        yf = gen(x)
                        fk = disc(x, yf) if disc else None
                        fl = fk if isinstance(fk, list) else ([fk] if fk is not None else None)
                        lg, det = crit(fl, yf, y, seg)
                    opt_g.zero_grad(set_to_none=True)
                    sg.scale(lg).backward(); sg.unscale_(opt_g)
                    torch.nn.utils.clip_grad_norm_(gen.parameters(), self.cfg.grad_clip)
                    sg.step(opt_g); sg.update(); _ema_update()

                sum_g += lg.item(); n += 1
                pbar.set_postfix({'G': f'{lg.item():.3f}',
                                  'D': f'{sum_d/n:.4f}' if disc else '-',
                                  'lr': f'{opt_g.param_groups[0]["lr"]:.1e}'})

            sch_g.step()
            if sch_d: sch_d.step()
            avg_g, avg_d = sum_g/max(n,1), sum_d/max(n,1)

            if epoch % self.cfg.val_every == 0:
                stats = self._validate(ema, val_dl)
                sa = stats['all']['ssim']['mean']
                st = stats['tumour']['ssim'].get('mean', float('nan'))
                sp = stats['all']['psnr']['mean']
                print(f'[{epoch:3d}] G={avg_g:.3f} D={avg_d:.4f}  SSIM={sa:.4f}  PSNR={sp:.2f}dB')
                if sa > best_ssim + 5e-4:
                    best_ssim = sa
                    best_m = {'epoch': epoch, 'ssim_all': sa, 'ssim_tumour': st, 'psnr_all': sp}
                    self._save(epoch, gen, disc, opt_g, opt_d, best_m, sg=sg, sd=sd, ema=ema)
                    print(f'     * new best SSIM={best_ssim:.4f}')
                log_rows.append({'epoch': epoch, 'loss_g': avg_g, 'loss_d': avg_d,
                                 'ssim_all': sa, 'ssim_tumour': st, 'psnr': sp})
                pd.DataFrame(log_rows).to_csv(self._log_path, index=False)
                if stopper(sa):
                    print(f'[EarlyStopping] No improvement for {self.cfg.patience} checks.'); break
            else:
                print(f'[{epoch:3d}] G={avg_g:.3f}  D={avg_d:.4f}')

        print(f'\nBest SSIM = {best_ssim:.4f}  (epoch {best_m.get("epoch","-")})')
        return best_m


print('Trainer defined.')

In [ ]:
class Evaluator:
    def __init__(self, cfg): self.cfg = cfg

    def run(self, loaders):
        gen  = build_generator(self.cfg).to(DEVICE)
        ckpt = torch.load(self.cfg.out_dir / 'best.pth', map_location=DEVICE, weights_only=False)
        gen.load_state_dict(ckpt.get('gen_ema', ckpt['gen']))
        print('[Evaluator] Using', 'EMA' if 'gen_ema' in ckpt else 'raw', 'weights.')
        gen.eval()
        tr = MetricsTracker(data_range=self.cfg.metric_data_range)
        with torch.no_grad():
            for x, y, seg, meta in tqdm(loaders['test'], desc='Test eval', ncols=80):
                x = x.to(DEVICE)
                with autocast(device_type=AMP_DEVICE):
                    pred = gen(x)
                pn = pred.cpu().float().numpy(); yn = y.float().numpy()
                for b in range(pn.shape[0]):
                    ht = bool(meta['has_tumour'][b]) if isinstance(meta['has_tumour'], torch.Tensor) \
                         else meta['has_tumour'][b]
                    tr.update(pn[b,0], yn[b,0], ht)
        stats = tr.summary()
        print('\n-- Test-Set Results ---')
        for m in ('ssim','psnr','mae'):
            v = stats['all'][m]
            if v['n'] > 0: print(f'  {m:5s} = {v["mean"]:.4f} +/- {v["std"]:.4f}  (n={v["n"]})')
        return stats


class Visualiser:
    @staticmethod
    def data_sample(data_dir, pair, n_samples=4):
        """Display source/target pairs from the data directory."""
        files = sorted(Path(data_dir).glob(f'*_{pair}_*_w.nii.gz'))
        if not files: print(f'No files for pair {pair!r} in {data_dir}'); return
        idx = np.linspace(0, len(files)-1, min(n_samples, len(files)), dtype=int)
        sn, tn = PAIR_LABELS.get(pair, ('Source','Target'))
        fig, axes = plt.subplots(2, len(idx), figsize=(5*len(idx), 10))
        if len(idx) == 1: axes = axes.reshape(2,1)
        for col, i in enumerate(idx):
            vol = nib.load(str(files[i])).get_fdata().astype(np.float32)
            axes[0,col].imshow(vol[:,:,0], cmap='gray', vmin=-1, vmax=1)
            axes[0,col].set_title(f'{sn}\n{files[i].name[:28]}', fontsize=8); axes[0,col].axis('off')
            axes[1,col].imshow(vol[:,:,1], cmap='gray', vmin=-1, vmax=1)
            axes[1,col].set_title(tn, fontsize=8); axes[1,col].axis('off')
        plt.suptitle(f'MMIT-DDPM  |  {pair}  ({sn} -> {tn})', fontsize=11, fontweight='bold')
        plt.tight_layout(); plt.show()

    @staticmethod
    def prediction_grid(pid, z, x_np, pred_np, gt_np, save_path=None, pair='ft1ce'):
        """Source | GT | Prediction | Residual."""
        in_c = x_np.shape[0]
        sn, tn = PAIR_LABELS.get(pair, ('Source','Target'))
        fig, axes = plt.subplots(1, in_c+3, figsize=(5*(in_c+3), 5))
        for j in range(in_c):
            axes[j].imshow(x_np[j], cmap='gray', vmin=-1, vmax=1)
            axes[j].set_title(f'{sn} (input)'); axes[j].axis('off')
        axes[in_c].imshow(gt_np, cmap='gray', vmin=-1, vmax=1)
        axes[in_c].set_title(f'{tn} GT', color='limegreen'); axes[in_c].axis('off')
        axes[in_c+1].imshow(pred_np, cmap='gray', vmin=-1, vmax=1)
        axes[in_c+1].set_title(f'{tn} Pred', color='dodgerblue'); axes[in_c+1].axis('off')
        diff = pred_np - gt_np; vm = max(abs(diff.min()), abs(diff.max()), 1e-8)
        axes[in_c+2].imshow(diff, cmap='RdBu_r', vmin=-vm, vmax=vm)
        axes[in_c+2].set_title('Residual', color='tomato'); axes[in_c+2].axis('off')
        mask = gt_np != 0; dr = 2.0
        ssim = float(skimage_ssim(pred_np, gt_np, data_range=dr)) if mask.any() else 0.0
        psnr = float(skimage_psnr(gt_np, pred_np, data_range=dr)) if mask.any() else 0.0
        plt.suptitle(f'Patient {pid} | z={z}   SSIM={ssim:.4f}   PSNR={psnr:.2f}dB',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.show()

    @staticmethod
    def training_curves(log_csv):
        if not Path(log_csv).exists(): print(f'Log not found: {log_csv}'); return
        df = pd.read_csv(log_csv)
        fig, axes = plt.subplots(1, 3, figsize=(17, 5))
        axes[0].plot(df.epoch, df.loss_g, color='royalblue', label='L_G')
        axes[0].set(title='Generator Loss', xlabel='Epoch'); axes[0].legend()
        axes[1].plot(df.epoch, df.loss_d, color='tomato', label='L_D')
        axes[1].set(title='Discriminator Loss', xlabel='Epoch'); axes[1].legend()
        axes[2].plot(df.epoch, df.ssim_all, color='green', label='SSIM')
        axes[2].plot(df.epoch, df.psnr/50.0, color='purple', label='PSNR/50', ls='--')
        axes[2].set(title='Validation Metrics', xlabel='Epoch'); axes[2].legend()
        plt.suptitle(f'Training — {Path(log_csv).parent.name}', fontsize=12, fontweight='bold')
        plt.tight_layout(); plt.show()


print('Evaluator and Visualiser defined.')

---
## STEP 1 — Explore the data

In [ ]:
def scan_data(data_dir, pair):
    files = sorted(Path(data_dir).glob(f'*_{pair}_*_w.nii.gz'))
    pids  = {parse_mmit_filename(f.name)['patient_id']
             for f in files if parse_mmit_filename(f.name)}
    print(f'{Path(data_dir).name}: {len(files):,} slices | {len(pids)} patients | pair={pair}')
    if files:
        vol = nib.load(str(files[0])).get_fdata()
        print(f'  shape={vol.shape}  range=[{vol.min():.3f}, {vol.max():.3f}]')

print('-- Data summary ---')
scan_data(CFG.train_dir, CFG.translation_pair)
scan_data(CFG.test_dir,  CFG.translation_pair)

sn, tn = PAIR_LABELS.get(CFG.translation_pair, ('Source','Target'))
print(f"\nPair '{CFG.translation_pair}': {sn} -> {tn}")
Visualiser.data_sample(CFG.train_dir, CFG.translation_pair, n_samples=4)

## STEP 2 — Generate patient splits

In [ ]:
split_mgr = SplitManager(CFG)
split_mgr.make()          # skips if splits.csv already exists
splits = split_mgr.load()

## STEP 3 — Smoke test

Full pipeline validation on a small fraction. Should complete in < 5 min on T4.

In [ ]:
SMOKE_CFG = Config(
    translation_pair = CFG.translation_pair,
    in_c             = CFG.in_c,
    model_type       = 'attn_cgan',
    dataset_fraction = 0.5,
    max_epochs       = 3,
    batch_size       = 4,
    val_every        = 1,
    patience         = 99,
    num_workers      = 0,
    out_name         = 'smoke_test',
)
print(f'Smoke test: pair={SMOKE_CFG.translation_pair}  model={SMOKE_CFG.model_type}  '
      f'frac={SMOKE_CFG.dataset_fraction:.0%}  epochs={SMOKE_CFG.max_epochs}')

smoke_splits  = SplitManager(SMOKE_CFG).load()
smoke_loaders = DataLoaderFactory(SMOKE_CFG).build(smoke_splits)
smoke_results = Trainer(SMOKE_CFG).fit(smoke_loaders)

print('\n-- Smoke test PASSED --')
print('All components (data -> model -> loss -> backward -> checkpoint -> eval) OK.')
print(f'Results: {smoke_results}')

## STEP 4 — Full training

In [ ]:
CFG.model_type       = 'attn_cgan'    # unet | pix2pix | attn_cgan
CFG.max_epochs       = 200
CFG.batch_size       = 16
CFG.val_every        = 5
CFG.num_workers      = 2
CFG.dataset_fraction = 1.0
CFG.out_name         = f'run_{CFG.translation_pair}_{CFG.model_type}'

full_splits  = SplitManager(CFG).load()
full_loaders = DataLoaderFactory(CFG).build(full_splits)

results = Trainer(CFG).fit(full_loaders, resume=False)
print('\nFull training complete:', results)

## STEP 5 — Test-set evaluation

In [ ]:
if not (CFG.out_dir / 'best.pth').exists():
    raise FileNotFoundError(f'No checkpoint at {CFG.out_dir}/best.pth — run STEP 4 first.')

test_stats = Evaluator(CFG).run(full_loaders)

## STEP 6 — Visualize results

In [ ]:
Visualiser.training_curves(str(CFG.out_dir / 'train_log.csv'))

ckpt_path = CFG.out_dir / 'best.pth'
if not ckpt_path.exists():
    raise FileNotFoundError(f'No checkpoint at {ckpt_path} — run STEP 4 first.')

gen_eval = build_generator(CFG).to(DEVICE)
ckpt     = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
gen_eval.load_state_dict(ckpt.get('gen_ema', ckpt['gen']))
gen_eval.eval()

test_info = full_splits.get('test', full_splits.get('val', {}))
test_ds   = MMITBraTS2DDataset(
    data_dir    = test_info['data_dir'],
    patient_ids = test_info['pids'],
    pair        = CFG.translation_pair,
    augment     = False,
    cfg         = CFG,
)

if len(test_ds) == 0:
    print('No test slices found.')
else:
    x_t, y_t, seg_t, meta_t = test_ds[len(test_ds) // 2]
    with torch.no_grad():
        with autocast(device_type=AMP_DEVICE):
            pred_t = gen_eval(x_t.unsqueeze(0).to(DEVICE))[0, 0].cpu().float().numpy()
    Visualiser.prediction_grid(
        pid=meta_t['pid'], z=meta_t['z'],
        x_np=x_t.numpy(), pred_np=pred_t, gt_np=y_t[0].numpy(),
        save_path=str(CFG.out_dir / 'prediction_grid.png'),
        pair=CFG.translation_pair,
    )